# Step 3 — Flight data and information settings

Cleans the BTS on-time records, computes exact UTC departure/arrival times, and attaches weather under the oracle and deployable information settings (paper Sect. 3.1–3.2).

- **Input:** `data/raw/bts/Combined_Flights_<year>.parquet`, `data/processed/weather_hourly_utc.parquet`
- **Output:** `data/processed/flights_clean.parquet` and `data/processed/wx_{oracle,H0,H2,H6}.parquet` (joined by `row_id`)

Part A cleans flights and fixes time alignment; Part B builds the weather feature sets.

## Part A — Flight cleaning and UTC alignment

### Imports

In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import airportsdata
import holidays

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'README.md').exists())  # project root

pd.set_option('display.max_columns', 50)
OUT_DIR = ROOT / 'data' / 'processed'; OUT_DIR.mkdir(parents=True, exist_ok=True)
TOP_10_STATES = ['CA', 'TX', 'FL', 'IL', 'GA', 'NY', 'CO', 'NC', 'VA', 'WA']

### Load flights in the study area
Only the 20 required columns are read, so post-departure fields (`DepDelay`, `TaxiOut`, `WheelsOff`, …) never enter the modelling table. Flights whose origin **and** destination lie in the ten study states are kept. Both the target `ArrDelayMinutes` = max(0, `ArrDelay`) and the signed `ArrDelay` are retained.

In [2]:
COLS = ['FlightDate', 'Airline', 'Origin', 'Dest', 'Cancelled', 'Diverted',
        'CRSDepTime', 'CRSArrTime', 'CRSElapsedTime', 'Distance', 'DistanceGroup',
        'Month', 'DayOfWeek', 'DepTimeBlk', 'OriginState', 'DestState', 'Tail_Number',
        'ArrDelay', 'ArrDelayMinutes', 'ArrDel15']

frames = []
for f in sorted(glob.glob(str(ROOT / 'data' / 'raw' / 'bts' / 'Combined_Flights_*.parquet'))):
    d = pd.read_parquet(f, columns=COLS)
    d = d[d['OriginState'].isin(TOP_10_STATES) & d['DestState'].isin(TOP_10_STATES)]
    frames.append(d)
    print(f'  {f}: {len(d):,} flights')
flight = pd.concat(frames, ignore_index=True)
del frames
flight['FlightDate'] = pd.to_datetime(flight['FlightDate'])
print(f'Total: {len(flight):,} flights')

  c:\Users\Admin\Downloads\on-going assement 1\source_code - Copy\data\raw\bts\Combined_Flights_2018.parquet: 1,998,315 flights
  c:\Users\Admin\Downloads\on-going assement 1\source_code - Copy\data\raw\bts\Combined_Flights_2019.parquet: 2,845,280 flights
  c:\Users\Admin\Downloads\on-going assement 1\source_code - Copy\data\raw\bts\Combined_Flights_2020.parquet: 1,749,407 flights
  c:\Users\Admin\Downloads\on-going assement 1\source_code - Copy\data\raw\bts\Combined_Flights_2021.parquet: 2,225,508 flights
  c:\Users\Admin\Downloads\on-going assement 1\source_code - Copy\data\raw\bts\Combined_Flights_2022.parquet: 1,482,849 flights
Total: 10,301,359 flights


### Remove cancelled, diverted, and invalid records
Drops cancelled or diverted flights, missing labels, the invalid code `USA`, and non-positive scheduled elapsed times.

In [3]:
n0 = len(flight)
flight = flight[(flight['Cancelled'] == False) & (flight['Diverted'] == False)]
flight = flight.dropna(subset=['ArrDelayMinutes', 'ArrDel15', 'CRSElapsedTime'])
flight = flight[(flight['Origin'] != 'USA') & (flight['Dest'] != 'USA')]
flight = flight[flight['CRSElapsedTime'] > 0]
flight = flight.drop(columns=['Cancelled', 'Diverted']).reset_index(drop=True)
print(f'{n0:,} -> {len(flight):,} flights')
print(flight['FlightDate'].dt.year.value_counts().sort_index())

10,301,359 -> 9,981,341 flights
FlightDate
2018    1960600
2019    2777881
2020    1635896
2021    2175957
2022    1431007
Name: count, dtype: int64


### Calendar features
Scheduled hours, cyclic encodings, weekend, peak-hour and US-holiday flags. `CRSArrTime = 2400` is mapped to hour 0.

In [4]:
hhmm = flight['CRSDepTime'].astype(int)
flight['CRSDepHour'] = (hhmm // 100).clip(upper=23).astype('int8')
flight['CRSArrHour'] = (flight['CRSArrTime'].astype(int) // 100 % 24).astype('int8')
flight['DepHour_sin'] = np.sin(2 * np.pi * flight['CRSDepHour'] / 24)
flight['DepHour_cos'] = np.cos(2 * np.pi * flight['CRSDepHour'] / 24)
flight['Month_sin'] = np.sin(2 * np.pi * flight['Month'] / 12)
flight['Month_cos'] = np.cos(2 * np.pi * flight['Month'] / 12)
flight['DoW_sin'] = np.sin(2 * np.pi * flight['DayOfWeek'] / 7)
flight['DoW_cos'] = np.cos(2 * np.pi * flight['DayOfWeek'] / 7)
flight['IsWeekend'] = flight['DayOfWeek'].isin([6, 7]).astype('int8')
flight['IsPeakHour'] = flight['CRSDepHour'].isin([7, 8, 9, 17, 18, 19]).astype('int8')
us_hol = holidays.US(years=sorted(flight['FlightDate'].dt.year.unique().tolist()))
flight['IsHoliday'] = flight['FlightDate'].dt.date.isin(us_hol).astype('int8')
print('calendar features done')

calendar features done


### Departure and arrival times in UTC
- `dep_utc`: scheduled departure converted to UTC with the origin airport's time zone, kept at minute resolution.
- `arr_utc = dep_utc + CRSElapsedTime`: correct across midnight and time zones. (Concatenating the flight date with the scheduled arrival hour misplaces overnight arrivals by 24 h.)
- Non-existent local times at the DST transition are shifted forward.

In [5]:
ap = airportsdata.load('IATA')
tz_of = {c: ap.get(c, {}).get('tz') for c in set(flight['Origin']) | set(flight['Dest'])}
print('Airports without time zone:', [c for c, t in tz_of.items() if t is None])

dep_local = flight['FlightDate'] + pd.to_timedelta((hhmm // 100) * 60 + hhmm % 100, unit='m')
dep_utc = pd.Series(pd.NaT, index=flight.index, dtype='datetime64[ns]')
for code, idx in flight.groupby('Origin').groups.items():
    tz = tz_of.get(code)
    if tz is None:
        continue
    loc = dep_local.loc[idx].dt.tz_localize(tz, ambiguous='NaT', nonexistent='shift_forward')
    dep_utc.loc[idx] = loc.dt.tz_convert('UTC').dt.tz_localize(None).values

flight['dep_utc'] = dep_utc
flight['arr_utc'] = flight['dep_utc'] + pd.to_timedelta(flight['CRSElapsedTime'], unit='m')

n1 = len(flight)
flight = flight.dropna(subset=['dep_utc', 'arr_utc']).reset_index(drop=True)
print(f'Dropped {n1 - len(flight):,} flights with ambiguous local time (DST fall-back)')

Airports without time zone: []
Dropped 10 flights with ambiguous local time (DST fall-back)


### Validation of arrival times
`arr_utc` is converted back to the destination's local time and compared with the published `CRSArrTime`; the share of flights arriving on a later calendar day is also reported (paper Sect. 4.2).

In [6]:
arr_hhmm = pd.Series(-1, index=flight.index)
arr_date = pd.Series(pd.NaT, index=flight.index, dtype='datetime64[ns]')
for code, idx in flight.groupby('Dest').groups.items():
    tz = tz_of.get(code)
    if tz is None:
        continue
    loc = flight.loc[idx, 'arr_utc'].dt.tz_localize('UTC').dt.tz_convert(tz)
    arr_hhmm.loc[idx] = (loc.dt.hour * 100 + loc.dt.minute).values
    arr_date.loc[idx] = loc.dt.tz_localize(None).dt.normalize().values

crs_arr = flight['CRSArrTime'].astype(int) % 2400
diff = ((arr_hhmm // 100 * 60 + arr_hhmm % 100) - (crs_arr // 100 * 60 + crs_arr % 100)) % 1440
diff = np.minimum(diff, 1440 - diff)
print(f'Computed arrival matches CRSArrTime (<= 1 min): {(diff <= 1).mean():.3%}')

flight['arr_overnight'] = (arr_date != flight['FlightDate']).astype('int8')
print(f'Arrivals on a later date than FlightDate (misaligned by naive join): '
      f'{flight["arr_overnight"].sum():,} ({flight["arr_overnight"].mean():.2%})')

Computed arrival matches CRSArrTime (<= 1 min): 99.998%
Arrivals on a later date than FlightDate (misaligned by naive join): 262,561 (2.63%)


### Save the flight table
Writes `data/processed/flights_clean.parquet` with a `row_id` used to join the weather feature sets.

In [7]:
flight['row_id'] = np.arange(len(flight), dtype=np.int64)
float_cols = flight.select_dtypes('float64').columns
flight[float_cols] = flight[float_cols].astype('float32')
flight.to_parquet(OUT_DIR / 'flights_clean.parquet', index=False)
print(f'saved {len(flight):,} flights x {flight.shape[1]} columns')
print(f'ArrDelayMinutes (target): mean={flight["ArrDelayMinutes"].mean():.2f}, median={flight["ArrDelayMinutes"].median():.1f}')
print(f'ArrDelay (signed):        mean={flight["ArrDelay"].mean():.2f}, median={flight["ArrDelay"].median():.1f}')

saved 9,981,331 flights x 33 columns
ArrDelayMinutes (target): mean=12.92, median=0.0
ArrDelay (signed):        mean=3.77, median=-7.0


## Part B — Weather under different information settings

A prediction for a flight with scheduled departure `t_dep` is issued at `t0 = t_dep − H`.

| Setting | Origin weather | Destination weather | Deployable |
|---|---|---|---|
| `oracle` | bucket containing `t_dep` | bucket containing `t_arr` | No (upper bound) |
| `H0`, `H2`, `H6` | last closed bucket before `t0` | same bucket, at the destination | Yes |

The last closed bucket is `b = floor(t0) − 1h`: bucket `b` contains reports in `[b, b+1h)` and is complete only at `b+1h`. Example: for `t0 = 14:35` the 13:00 bucket is used.

### Load weather and join helper
`attach` joins weather by (airport, hour) with a left join, preserving row order.

In [8]:
WX_FEATS = ['temp_f', 'dewpt_f', 'humidity', 'slp', 'pressure', 'visibility_mi',
            'ceiling_ft', 'cloud_max', 'wind_mph', 'gust_mph', 'gust_excess', 'has_gust',
            'wind_dir_sin', 'wind_dir_cos', 'precip_in',
            'has_rain', 'has_snow', 'has_fog', 'has_mist', 'has_thunder', 'has_thunder_vc',
            'has_freezing', 'is_heavy', 'flight_cat_ord', 'wx_missing',
            'precip_3h', 'vis_min_3h', 'ceil_min_3h', 'wind_max_3h', 'thunder_3h', 'slp_tend_3h']

weather = pd.read_parquet(OUT_DIR / 'weather_hourly_utc.parquet', columns=['iata', 'obs_hour_utc'] + WX_FEATS)
weather['obs_hour_utc'] = weather['obs_hour_utc'].astype('datetime64[ns]')
keys = flight[['row_id', 'Origin', 'Dest', 'dep_utc', 'arr_utc']].copy()
keys['dep_utc'] = keys['dep_utc'].astype('datetime64[ns]')
keys['arr_utc'] = keys['arr_utc'].astype('datetime64[ns]')


def attach(airport_col, hour_values, prefix):
    k = pd.DataFrame({airport_col: keys[airport_col].values, 'h': hour_values})
    w = weather.rename(columns={'iata': airport_col, 'obs_hour_utc': 'h',
                                **{c: prefix + c for c in WX_FEATS}})
    out = k.merge(w, on=[airport_col, 'h'], how='left')
    assert len(out) == len(k)
    return out[[prefix + c for c in WX_FEATS]]


def build(setting, org_hours, dst_hours, extra=None):
    o = attach('Origin', org_hours, 'org_')
    d = attach('Dest', dst_hours, 'dst_')
    res = pd.concat([keys[['row_id']].reset_index(drop=True), o, d], axis=1)
    for name, val in (extra or {}).items():
        res[name] = np.asarray(val, dtype='float32')
    res.to_parquet(OUT_DIR / f'wx_{setting}.parquet', index=False)
    print(f'wx_{setting}: matched origin {o.iloc[:, 0].notna().mean():.2%}, dest {d.iloc[:, 0].notna().mean():.2%}')

### Oracle setting
Realized weather at the scheduled departure (origin) and arrival (destination) hours. Not available at prediction time; used as an upper bound.

In [9]:
build('oracle',
      keys['dep_utc'].dt.floor('h').values,
      keys['arr_utc'].dt.floor('h').values)

wx_oracle: matched origin 99.99%, dest 99.99%


### Deployable settings `H0`, `H2`, `H6`
Both endpoints use the bucket `floor(t0) − 1h`; `dst_lead_h = (t_arr − t0)` in hours tells the model how far ahead of arrival the destination information is. Horizons are set in `HORIZONS_H`.

In [10]:
HORIZONS_H = [0, 2, 6]
for H in HORIZONS_H:
    t0 = keys['dep_utc'] - pd.Timedelta(hours=H)
    last_complete = (t0.dt.floor('h') - pd.Timedelta(hours=1)).values
    lead_h = (keys['arr_utc'] - t0).dt.total_seconds() / 3600
    build(f'H{H}', last_complete, last_complete, extra={'dst_lead_h': lead_h})

# Leakage check: the H0 bucket must close at or before t0
t0 = keys['dep_utc']
assert ((t0.dt.floor('h') - pd.Timedelta(hours=1)) + pd.Timedelta(hours=1) <= t0).all()
print('OK: no bucket extends beyond the issuance time')

wx_H0: matched origin 99.99%, dest 99.99%
wx_H2: matched origin 99.98%, dest 99.98%
wx_H6: matched origin 99.97%, dest 99.97%
OK: no bucket extends beyond the issuance time


### Sanity check
Rank correlations between weather and delay under `oracle` and `H2`; correlations are expected to be weaker for `H2`.

In [11]:
y = flight['ArrDelayMinutes'].values
for setting in ['oracle', 'H2']:
    W = pd.read_parquet(OUT_DIR / f'wx_{setting}.parquet',
                        columns=['org_visibility_mi', 'org_precip_in', 'dst_visibility_mi', 'dst_precip_in', 'dst_thunder_3h'])
    print(f'\n[{setting}] correlation with ArrDelayMinutes:')
    for c in W.columns:
        print(f'  {c:22s} r = {np.corrcoef(W[c].fillna(W[c].median()), y)[0, 1]:+.4f}')


[oracle] correlation with ArrDelayMinutes:
  org_visibility_mi      r = -0.0633
  org_precip_in          r = +0.0776
  dst_visibility_mi      r = -0.0509
  dst_precip_in          r = +0.0553
  dst_thunder_3h         r = +0.0821

[H2] correlation with ArrDelayMinutes:
  org_visibility_mi      r = -0.0503
  org_precip_in          r = +0.0478
  dst_visibility_mi      r = -0.0415
  dst_precip_in          r = +0.0385
  dst_thunder_3h         r = +0.0586
